# 22_Packet_P005_Robust_Lab_Panel
## Materia Arche Agent OS v3.0

### Correction Notice (P-004 / NB21)
P-004 used full-dataset rankings (including training data) to assess rank stability. When a device is in the training set, the model memorizes it and ranks it artificially high. The `mean_rank` metric was therefore inflated for devices frequently in the training set. This notebook uses **test-set-only rankings** to avoid this evaluation bias.

---

**Work Packet ID:** P-005
**Decision this packet informs:** Can we produce a new, split-robust 3-candidate shortlist for Phase 2?
**Fastest falsifier:** Run 20 random splits, rank only test-set devices each time — if no 3 devices appear in top-20 in ≥80% of their test appearances, rankings are too noisy for a stable panel.
**Success threshold:** 3 devices with top-20 test appearance rate ≥80% and actual T80 ≥500h
**Failure / kill threshold:** Fewer than 3 devices meet criteria
**Reuse requirement if it fails:** Document best available candidates with honest uncertainty context
**Dataset ID/version:** perovskite_stability_clean.csv (1,543 devices)
**Benchmark ID:** P-004 test-set frequency data
**Owner:** ML Scientist
**Reviewer:** Evidence Guardian
**Release ceiling:** Internal

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from scipy.stats import kendalltau
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded — Packet P-005")

Libraries loaded — Packet P-005


## 1. Load data and locked model config

In [2]:
df = pd.read_csv("perovskite_stability_clean.csv")
print(f"Dataset: {len(df)} devices")

ml_features = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
               'MA', 'FA', 'Cs',
               'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
               'Perovskite_thickness', 'Cell_area_measured', 'JV_default_Voc',
               'JV_default_Jsc', 'JV_default_FF']

X = df[ml_features].fillna(0)
y = np.log1p(df['Stability_PCE_T80'])

# Locked ET config from NB15
et_params = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False)

print(f"Features: {len(ml_features)}")
print(f"Target: log1p(Stability_PCE_T80)")

Dataset: 1543 devices
Features: 16
Target: log1p(Stability_PCE_T80)


## 2. Test-set-only ranking across 20 splits

For each split: train on 80%, predict and rank only the 20% test set. Track how often each device appears in the test-set top-20 when it's held out.

In [3]:
n_splits = 20
seeds = list(range(1, n_splits + 1))

# Track per-device: times in test set, times ranked top-20 in test set, all test ranks
device_test_appearances = Counter()
device_top20_counts = Counter()
device_test_ranks = {idx: [] for idx in range(len(df))}  # only test-set ranks

split_taus = []  # track model quality per split

for seed in seeds:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    et = ExtraTreesRegressor(random_state=42, **et_params)
    et.fit(X_tr, y_tr)
    
    # Predict and rank TEST SET ONLY
    pred_test = et.predict(X_te)
    tau, _ = kendalltau(y_te, pred_test)
    split_taus.append(tau)
    
    # Rank within test set
    test_ranks = pd.Series(pred_test, index=X_te.index).rank(ascending=False).astype(int)
    
    for idx in X_te.index:
        device_test_appearances[idx] += 1
        device_test_ranks[idx].append(test_ranks[idx])
        if test_ranks[idx] <= 20:
            device_top20_counts[idx] += 1

print(f"Completed {n_splits} splits")
print(f"Mean tau-b across splits: {np.mean(split_taus):.4f} +/- {np.std(split_taus):.4f}")
print(f"Test set size per split: ~{len(X_te)} devices")

Completed 20 splits
Mean tau-b across splits: 0.3025 +/- 0.0418
Test set size per split: ~309 devices


## 3. Compute test-set rank stability for all devices

In [4]:
# Build stability table for devices that appeared in at least 3 test sets
rows = []
for idx in range(len(df)):
    n_appearances = device_test_appearances[idx]
    if n_appearances < 3:
        continue
    n_top20 = device_top20_counts[idx]
    ranks = device_test_ranks[idx]
    rows.append({
        'device_idx': idx,
        'actual_T80': np.expm1(y.iloc[idx]),
        'test_appearances': n_appearances,
        'top20_count': n_top20,
        'top20_rate': n_top20 / n_appearances,
        'mean_test_rank': np.mean(ranks),
        'std_test_rank': np.std(ranks),
        'min_test_rank': np.min(ranks),
        'max_test_rank': np.max(ranks),
    })

stability = pd.DataFrame(rows).set_index('device_idx')
stability = stability.sort_values('top20_rate', ascending=False)

print(f"Devices with ≥3 test appearances: {len(stability)}")
print(f"Devices with 100% top-20 rate: {(stability['top20_rate'] == 1.0).sum()}")
print(f"Devices with ≥80% top-20 rate: {(stability['top20_rate'] >= 0.8).sum()}")

Devices with ≥3 test appearances: 1223
Devices with 100% top-20 rate: 34
Devices with ≥80% top-20 rate: 43


## 4. Candidate selection

Criteria:
- Top-20 test appearance rate ≥80%
- Actual T80 ≥500h (meaningful stability — we need candidates worth testing)
- At least 3 test appearances (statistical minimum)

In [5]:
# Filter candidates
candidates = stability[
    (stability['top20_rate'] >= 0.80) &
    (stability['actual_T80'] >= 500) &
    (stability['test_appearances'] >= 3)
].copy()

# Sort by top20_rate descending, then by mean_test_rank ascending
candidates = candidates.sort_values(['top20_rate', 'mean_test_rank'], ascending=[False, True])

print("=" * 85)
print("CANDIDATES MEETING CRITERIA (top-20 rate ≥80%, actual T80 ≥500h, ≥3 appearances)")
print("=" * 85)
print(f"{'Idx':>5} {'Actual T80':>10} {'Appearances':>12} {'Top-20':>7} {'Rate':>6} {'Mean Rank':>10} {'Std':>6} {'Range':>12}")
print("-" * 85)

for idx, row in candidates.iterrows():
    rng = f"[{row['min_test_rank']:.0f}, {row['max_test_rank']:.0f}]"
    print(f"{idx:>5} {row['actual_T80']:>10.0f} {row['test_appearances']:>12.0f} {row['top20_count']:>7.0f} {row['top20_rate']:>6.0%} {row['mean_test_rank']:>10.1f} {row['std_test_rank']:>6.1f} {rng:>12}")

print(f"\nTotal qualifying candidates: {len(candidates)}")

CANDIDATES MEETING CRITERIA (top-20 rate ≥80%, actual T80 ≥500h, ≥3 appearances)
  Idx Actual T80  Appearances  Top-20   Rate  Mean Rank    Std        Range
-------------------------------------------------------------------------------------
  850       3400            4       4   100%        1.0    0.0       [1, 1]
 1374       1400            3       3   100%        1.7    0.5       [1, 2]
  848       2200            3       3   100%        2.0    0.8       [1, 3]
 1380       1824            7       7   100%        2.1    1.1       [1, 4]
  398       2240            5       5   100%        2.8    0.7       [2, 4]
  849       3400            5       5   100%        3.2    1.2       [2, 5]
 1066       5400            6       6   100%        3.3    3.6      [1, 11]
 1064       5400            5       5   100%        3.4    3.0       [1, 9]
  397       1600            3       3   100%        3.7    1.7       [2, 6]
  121        656            4       4   100%        4.2    4.0      [1, 1

## 5. Select robust lab panel

In [6]:
# Select top 3 (or fewer if not enough candidates)
n_panel = min(3, len(candidates))
panel = candidates.head(n_panel).copy()

# Add per-tree prediction intervals from frozen split for context
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
et = ExtraTreesRegressor(random_state=42, **et_params)
et.fit(X_tr, y_tr)
tree_preds_all = np.array([tree.predict(X) for tree in et.estimators_])
pred_mean_all = np.expm1(np.mean(tree_preds_all, axis=0))
pred_p10_all = np.expm1(np.percentile(tree_preds_all, 10, axis=0))
pred_p90_all = np.expm1(np.percentile(tree_preds_all, 90, axis=0))

panel['pred_mean_hours'] = [pred_mean_all[idx] for idx in panel.index]
panel['pred_p10_hours'] = [pred_p10_all[idx] for idx in panel.index]
panel['pred_p90_hours'] = [pred_p90_all[idx] for idx in panel.index]

# Add composition info
comp_cols = ['Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl', 'MA', 'FA', 'Cs']
for col in comp_cols:
    panel[col] = [df.iloc[idx][col] for idx in panel.index]

print("=" * 85)
print(f"ROBUST LAB PANEL — {n_panel} CANDIDATES")
print("=" * 85)

for i, (idx, row) in enumerate(panel.iterrows(), 1):
    print(f"\n  Candidate #{i} — Device {idx}")
    print(f"    Actual T80: {row['actual_T80']:.0f}h")
    print(f"    Test-set top-20: {row['top20_count']:.0f}/{row['test_appearances']:.0f} appearances ({row['top20_rate']:.0%})")
    print(f"    Mean test rank: {row['mean_test_rank']:.1f} +/- {row['std_test_rank']:.1f} [{row['min_test_rank']:.0f}, {row['max_test_rank']:.0f}]")
    print(f"    Prediction: {row['pred_mean_hours']:.0f}h (80% CI: {row['pred_p10_hours']:.0f}–{row['pred_p90_hours']:.0f}h)")
    print(f"    Composition: Pb={row['Pb']:.2f} Sn={row['Sn']:.2f} | I={row['I']:.2f} Br={row['Br']:.2f} Cl={row['Cl']:.2f} | MA={row['MA']:.2f} FA={row['FA']:.2f} Cs={row['Cs']:.2f}")
    print(f"    Bandgap: {row['Perovskite_band_gap']:.2f} eV")

ROBUST LAB PANEL — 3 CANDIDATES

  Candidate #1 — Device 850
    Actual T80: 3400h
    Test-set top-20: 4/4 appearances (100%)
    Mean test rank: 1.0 +/- 0.0 [1, 1]
    Prediction: 3118h (80% CI: 2735–3400h)
    Composition: Pb=4.00 Sn=0.00 | I=13.00 Br=0.00 Cl=0.00 | MA=3.00 FA=0.00 Cs=0.00
    Bandgap: nan eV

  Candidate #2 — Device 1374
    Actual T80: 1400h
    Test-set top-20: 3/3 appearances (100%)
    Mean test rank: 1.7 +/- 0.5 [1, 2]
    Prediction: 2235h (80% CI: 526–5000h)
    Composition: Pb=1.00 Sn=0.00 | I=3.00 Br=0.00 Cl=0.00 | MA=1.00 FA=0.00 Cs=0.00
    Bandgap: nan eV

  Candidate #3 — Device 848
    Actual T80: 2200h
    Test-set top-20: 3/3 appearances (100%)
    Mean test rank: 2.0 +/- 0.8 [1, 3]
    Prediction: 2214h (80% CI: 1795–2735h)
    Composition: Pb=4.00 Sn=0.00 | I=13.00 Br=0.00 Cl=0.00 | MA=3.00 FA=0.00 Cs=0.00
    Bandgap: nan eV


## 6. Compare old vs new panel

In [7]:
# Old frozen top-3 from NB17
old_panel = [1063, 1374, 289]

print("=" * 75)
print("OLD vs NEW LAB PANEL")
print("=" * 75)
print("")
print("OLD PANEL (NB17 frozen split, seed=42):")
for idx in old_panel:
    if idx in stability.index:
        s = stability.loc[idx]
        print(f"  Device {idx}: T80={s['actual_T80']:.0f}h, top-20 in {s['top20_count']:.0f}/{s['test_appearances']:.0f} ({s['top20_rate']:.0%}), mean rank {s['mean_test_rank']:.1f} +/- {s['std_test_rank']:.1f}")
    else:
        print(f"  Device {idx}: fewer than 3 test appearances")

print("")
print("NEW PANEL (cross-split robust):")
for idx, row in panel.iterrows():
    print(f"  Device {idx}: T80={row['actual_T80']:.0f}h, top-20 in {row['top20_count']:.0f}/{row['test_appearances']:.0f} ({row['top20_rate']:.0%}), mean rank {row['mean_test_rank']:.1f} +/- {row['std_test_rank']:.1f}")

# Export
export_cols = ['actual_T80', 'test_appearances', 'top20_count', 'top20_rate',
               'mean_test_rank', 'std_test_rank', 'min_test_rank', 'max_test_rank',
               'pred_mean_hours', 'pred_p10_hours', 'pred_p90_hours'] + comp_cols
panel[export_cols].round(2).to_csv("Packet_P005_Robust_Lab_Panel.csv")
print("\nPacket_P005_Robust_Lab_Panel.csv exported")

OLD vs NEW LAB PANEL

OLD PANEL (NB17 frozen split, seed=42):
  Device 1063: T80=5400h, top-20 in 3/4 (75%), mean rank 9.0 +/- 9.5
  Device 1374: T80=1400h, top-20 in 3/3 (100%), mean rank 1.7 +/- 0.5
  Device 289: T80=2300h, top-20 in 5/6 (83%), mean rank 9.2 +/- 10.7

NEW PANEL (cross-split robust):
  Device 850: T80=3400h, top-20 in 4/4 (100%), mean rank 1.0 +/- 0.0
  Device 1374: T80=1400h, top-20 in 3/3 (100%), mean rank 1.7 +/- 0.5
  Device 848: T80=2200h, top-20 in 3/3 (100%), mean rank 2.0 +/- 0.8

Packet_P005_Robust_Lab_Panel.csv exported


## 7. Honest status footer

In [8]:
# Determine status
if n_panel == 3 and all(panel['top20_rate'] >= 0.80):
    status = "Confirmed"
    decision = "Advance"
elif n_panel >= 2:
    status = "Promising"
    decision = "Iterate"
elif n_panel >= 1:
    status = "Promising"
    decision = "Iterate"
else:
    status = "Negative"
    decision = "Stop"

print("=" * 65)
print("HONEST STATUS — MATERIA ARCHE v3.0")
print("=" * 65)
print(f"Packet ID: P-005")
print(f"Status: {status}")
print(f"Evidence level: E3 (decision-grade shortlist)")
print(f"Decision outcome: {decision}")
print(f"Release ceiling: Internal")
print(f"Domain: perovskite")
print(f"Dataset version: perovskite_stability_clean.csv (1,543 devices)")
print(f"Benchmark: P-004 test-set frequency data")
print(f"This run: {n_panel} candidates selected with ≥80% top-20 test rate and T80 ≥500h")
print(f"  Model quality: mean tau-b {np.mean(split_taus):.4f} +/- {np.std(split_taus):.4f} across {n_splits} splits")

if n_panel == 3:
    print(f"What worked: Cross-split test-set ranking identifies stable candidates")
    print(f"What failed or remains uncertain: Prediction intervals remain wide (P-003); stability is measured, not predicted")
    print(f"What changes next: New panel replaces NB17 frozen panel for Phase 2 outreach")
elif n_panel >= 1:
    print(f"What worked: Found {n_panel} stable candidates")
    print(f"What failed or remains uncertain: Could not fill full 3-candidate panel with ≥80% rate + T80 ≥500h")
    print(f"What changes next: Consider relaxing criteria or adding more data")
else:
    print(f"What worked: Confirmed that no candidates are sufficiently rank-stable")
    print(f"What failed or remains uncertain: Dataset too noisy for stable shortlisting")
    print(f"What changes next: Collect more data before attempting Phase 2")

print(f"Reusable asset created: Packet_P005_Robust_Lab_Panel.csv")
print(f"Safeguard added: Test-set-only ranking prevents train-set memorization bias")
print(f"")
print(f"Reviewer sign-off: Evidence Guardian __________")

HONEST STATUS — MATERIA ARCHE v3.0
Packet ID: P-005
Status: Confirmed
Evidence level: E3 (decision-grade shortlist)
Decision outcome: Advance
Release ceiling: Internal
Domain: perovskite
Dataset version: perovskite_stability_clean.csv (1,543 devices)
Benchmark: P-004 test-set frequency data
This run: 3 candidates selected with ≥80% top-20 test rate and T80 ≥500h
  Model quality: mean tau-b 0.3025 +/- 0.0418 across 20 splits
What worked: Cross-split test-set ranking identifies stable candidates
What failed or remains uncertain: Prediction intervals remain wide (P-003); stability is measured, not predicted
What changes next: New panel replaces NB17 frozen panel for Phase 2 outreach
Reusable asset created: Packet_P005_Robust_Lab_Panel.csv
Safeguard added: Test-set-only ranking prevents train-set memorization bias

Reviewer sign-off: Evidence Guardian __________
